# Tidy Drive into the per-phase layout

Four cells. Run them in order. **Nothing moves until you set `APPLY = True`
in Step 3.**

This notebook is **self-contained on purpose**: it imports nothing from the
project. `scripts/inventory.py` and `scripts/organise_phases.py` do the same
job from a shell, but they live *inside* `phase1_3_repo.zip` -- so using them
would mean extracting a checkout into the very folder you are trying to
reorganise. A tool that tidies the folder a checkout lives in cannot depend on
that checkout existing.

The classification logic below is therefore a deliberate copy, and
`tests/test_scripts_organise.py` asserts the copy still agrees with
`scripts.inventory.classify` on a table of paths, so the two cannot drift
apart silently.

### Where you are going

```
My Drive/NeurIPS-CCAI-2026/
├── data/raw/*.nc         SHARED cubes -- NEVER moved
├── phase1_1/             checkout + notebook
├── phase1_2/             checkout; artefacts at data/phase1_2/{embeddings,masks}
└── phase1_3/             checkout
```

`data/raw` stays at the project root because the cubes are phase-independent:
1.2 and 1.3 both read them and scale-up will too. `data/paths.py` encodes the
same rule -- `RAW_DIR` is never phase-scoped and `reset_phase` refuses to clear
it. Filing them under `phase1_1/` would force every later phase to reach into
another phase's folder.

## Step 1: Mount Drive and find the project folder

Nothing is written in this cell.

In [ ]:
import glob, os

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
except ImportError:
    DRIVE = os.path.expanduser("~")
    print("not on Colab -- set PROJECT by hand below if this guess is wrong")

# Edit this if your folder is named or placed differently.
PROJECT = f"{DRIVE}/NeurIPS-CCAI-2026"

if not os.path.isdir(PROJECT):
    hits = [d for d in glob.glob(f"{DRIVE}/*/") + glob.glob(f"{DRIVE}/*/*/")
            if "NeurIPS" in d or "CCAI" in d]
    print(f"{PROJECT} does not exist. Candidates found on Drive:")
    for h in hits:
        print("   ", h)
    raise SystemExit("Set PROJECT to one of the paths above and re-run this cell.")

print(f"PROJECT = {PROJECT}")
print(f"top level: {sorted(os.listdir(PROJECT))}")

## Step 2: Inventory — list everything, classify every unit

Reads only. Also takes a **SHA-256 of every file**, which is what lets Step 4
prove the move lost nothing. On ~130 files this takes seconds; on a folder with
gigabytes of embeddings it can take a few minutes, and it is worth it once.

In [ ]:
import hashlib, json, re, shutil
from typing import NamedTuple

# === SHARED WITH scripts/inventory.py AND scripts/organise_phases.py -- BEGIN ===
# Pinned against those modules by tests/test_scripts_organise.py. If you change
# a rule here, change it there too; the test fails until they agree.

SKIP = {".git", "__pycache__", ".venv", ".pytest_cache", ".ipynb_checkpoints",
        ".DS_Store", ".mypy_cache", "node_modules"}

# The one path shared rather than owned by a phase. Mirrors data.paths.RAW_DIR
# and data.paths.reset_phase, which likewise refuse to phase-scope the cubes.
SHARED = ("data/raw",)

_PHASE_DIR = re.compile(r"^phase\d+_\d+$")
_PHASE_ZIP = re.compile(r"^(phase\d+_\d+)_repo\.zip$")


def classify(rel):
    """(kind, phase) for one unit path. The single decision in this notebook.

    kind:  shared | artefacts | bundle | phase_folder | checkout
    phase is None for shared units and for checkout units, whose owning phase
    is NOT decidable from the filesystem -- a checkout looks identical
    whichever bundle produced it.
    """
    rel = rel.replace(os.sep, "/").strip("/")
    if rel in SHARED:
        return "shared", None
    if rel.startswith("data/") and _PHASE_DIR.match(rel.split("/", 1)[1]):
        return "artefacts", rel.split("/", 1)[1]
    m = _PHASE_ZIP.match(os.path.basename(rel))
    if m and "/" not in rel:
        return "bundle", m.group(1)
    if _PHASE_DIR.match(rel):
        return "phase_folder", rel
    return "checkout", None


def destination(rel, kind, phase, checkout_phase):
    """Where one unit belongs, or None if it must stay put."""
    if kind == "shared":
        return None                       # never filed under a phase
    if kind == "phase_folder":
        return None                       # already correct
    if kind in ("artefacts", "bundle"):
        return f"{phase}/{rel}"
    if kind == "checkout":
        return None if checkout_phase is None else f"{checkout_phase}/{rel}"
    raise RuntimeError(f"unclassified unit {rel!r} (kind {kind!r})")

# === SHARED -- END ===


class Unit(NamedTuple):
    rel: str
    is_dir: bool
    kind: str
    phase: object
    n_files: int
    n_bytes: int


def _dir_stats(path):
    n = b = 0
    for dirpath, dirnames, filenames in os.walk(path):
        dirnames[:] = [d for d in dirnames if d not in SKIP]
        for f in filenames:
            if f in SKIP:
                continue
            n += 1
            try:
                b += os.path.getsize(os.path.join(dirpath, f))
            except OSError:
                pass
    return n, b


def iter_units(root):
    """Movable units: top-level entries, with data/ expanded ONE level.

    data/ is the exception because it holds three different kinds at once --
    shared cubes, phase artefacts and checkout code. Collapsing it into one
    unit would drag data/raw into a phase folder.
    """
    units = []
    for name in sorted(os.listdir(root)):
        if name in SKIP:
            continue
        full = os.path.join(root, name)
        if name == "data" and os.path.isdir(full):
            for sub in sorted(os.listdir(full)):
                if sub in SKIP:
                    continue
                rel, sf = f"data/{sub}", os.path.join(full, sub)
                d = os.path.isdir(sf)
                n, b = _dir_stats(sf) if d else (1, os.path.getsize(sf))
                k, p = classify(rel)
                units.append(Unit(rel, d, k, p, n, b))
            continue
        d = os.path.isdir(full)
        n, b = _dir_stats(full) if d else (1, os.path.getsize(full))
        k, p = classify(name)
        units.append(Unit(name, d, k, p, n, b))
    return units


def walk(root, with_hash=False):
    out = []
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = sorted(d for d in dirnames if d not in SKIP)
        for f in sorted(filenames):
            if f in SKIP:
                continue
            full = os.path.join(dirpath, f)
            rec = {"path": os.path.relpath(full, root).replace(os.sep, "/"),
                   "bytes": os.path.getsize(full)}
            if with_hash:
                h = hashlib.sha256()
                with open(full, "rb") as fh:
                    for blk in iter(lambda: fh.read(1 << 20), b""):
                        h.update(blk)
                rec["sha256"] = h.hexdigest()
            out.append(rec)
    return out


def human(n):
    for u in ("B", "kB", "MB", "GB"):
        if n < 1000 or u == "GB":
            return f"{n:.0f} {u}" if u == "B" else f"{n:.1f} {u}"
        n /= 1000.0


UNITS = iter_units(PROJECT)
w = max(len(u.rel) for u in UNITS)
print(f"{'UNIT'.ljust(w)}  {'KIND':<12} {'PHASE':<10} {'FILES':>7}  SIZE")
print("-" * (w + 42))
for u in UNITS:
    print(f"{u.rel.ljust(w)}  {u.kind:<12} {(u.phase or '-'):<10} "
          f"{u.n_files:>7}  {human(u.n_bytes)}")

print("\nhashing every file (this is the slow part)...")
BEFORE = walk(PROJECT, with_hash=True)
with open("/content/before.json" if os.path.isdir("/content") else "before.json",
          "w") as fh:
    json.dump(BEFORE, fh)
print(f"{len(UNITS)} units, {len(BEFORE)} files, "
      f"{human(sum(f['bytes'] for f in BEFORE))} total")

loose = [u.rel for u in UNITS if u.kind == "checkout"]
if loose:
    print(f"\n{len(loose)} checkout unit(s) have NO decidable phase -- a checkout "
          "looks the same\nwhichever bundle produced it, so Step 3 asks you "
          "rather than guessing:")
    print("  " + ", ".join(loose[:12]) + (" ..." if len(loose) > 12 else ""))

## Step 3: Plan, then apply

Set `CHECKOUT_PHASE` to whichever phase owns the loose code at the root — the
`data/*.py`, `encoders/`, `probes/`, `tests/`, `notebooks/` sitting directly in
the project folder. If your last upload was `phase1_2_repo.zip`, that is
`"phase1_2"`. Set it to `None` to leave the checkout where it is and move only
artefacts and bundles.

**`APPLY = False` prints the plan and moves nothing.** Read the plan, then set
it to `True` and re-run this cell.

In [ ]:
CHECKOUT_PHASE = "phase1_2"     # or "phase1_3", or None to leave it alone
APPLY = False                   # <-- set True once the plan below looks right


def plan(root, checkout_phase):
    moves = []
    for u in iter_units(root):
        dst = destination(u.rel, u.kind, u.phase, checkout_phase)
        if dst is None or dst == u.rel:
            continue
        moves.append((u.rel, dst))
    check(root, moves)
    return moves


def check(root, moves):
    """Every refusal, verified before a single file moves."""
    root_abs = os.path.abspath(root)
    seen = {}
    for src, dst in moves:
        for sh in SHARED:
            assert not (src == sh or src.startswith(sh + "/")), (
                f"REFUSED: would move {src!r}, but {sh!r} is SHARED across every "
                "phase and is never filed under one.")
        dst_abs = os.path.abspath(os.path.join(root_abs, dst))
        assert os.path.commonpath([root_abs, dst_abs]) == root_abs, (
            f"REFUSED: {dst!r} resolves outside the project folder.")
        assert dst not in seen, (
            f"REFUSED: {seen.get(dst)!r} and {src!r} both target {dst!r}.")
        seen[dst] = src
        assert not os.path.exists(dst_abs), (
            f"REFUSED: {dst!r} already exists. Inspect it first -- on Drive a "
            "same-named leftover from an older layout is common, and replacing "
            "a current artefact with a stale one is silent.")
        assert os.path.exists(os.path.join(root_abs, src)), (
            f"source vanished since Step 2: {src!r} -- re-run Step 2.")


MOVES = plan(PROJECT, CHECKOUT_PHASE)

print(f"checkout phase: {CHECKOUT_PHASE or '(none -- checkout stays put)'}\n")
if not MOVES:
    print("nothing to move: already in the phase layout.")
else:
    w = max(len(s) for s, _ in MOVES)
    print(f"{'FROM'.ljust(w)}  ->  TO")
    print("-" * (w + 40))
    for s, d in MOVES:
        print(f"{s.ljust(w)}  ->  {d}")
    print("-" * (w + 40))
    print(f"{len(MOVES)} move(s)")
    print("NOT moved, by contract: " + ", ".join(SHARED) + "  (shared)")

    if not APPLY:
        print("\n" + "=" * 66)
        print("DRY RUN -- nothing was moved.")
        print("Read the plan above, then set APPLY = True and re-run this cell.")
        print("=" * 66)
    else:
        check(PROJECT, MOVES)      # re-checked immediately before touching disk
        print()
        for s, d in MOVES:
            src, dst = os.path.join(PROJECT, s), os.path.join(PROJECT, d)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            assert not os.path.exists(dst), f"appeared mid-move: {d!r}"
            shutil.move(src, dst)
            print(f"[moved] {s}  ->  {d}")
        print(f"\n{len(MOVES)} unit(s) moved. Run Step 4 to verify.")

## Step 4: Verify nothing was lost

Re-inventories and compares SHA-256 sets against Step 2. Same hashes, different
paths, is the proof — on Drive an interrupted move is not obviously
distinguishable from a completed one, and this is what settles it.

In [ ]:
AFTER = walk(PROJECT, with_hash=True)

before_h = sorted(f["sha256"] for f in BEFORE)
after_h = sorted(f["sha256"] for f in AFTER)
print(f"files  before {len(BEFORE)}   after {len(AFTER)}")

if before_h == after_h:
    print("SHA-256 multiset IDENTICAL -- every file survived, contents unchanged")
else:
    lost = sorted(set(before_h) - set(after_h))
    gained = sorted(set(after_h) - set(before_h))
    bmap = {f["sha256"]: f["path"] for f in BEFORE}
    amap = {f["sha256"]: f["path"] for f in AFTER}
    print(f"\nMISMATCH: {len(lost)} hash(es) gone, {len(gained)} new")
    for h in lost[:10]:
        print(f"  lost   {bmap.get(h)}")
    for h in gained[:10]:
        print(f"  new    {amap.get(h)}")
    raise SystemExit("Do NOT continue. Investigate before running anything else.")

print()
w = max(len(u.rel) for u in iter_units(PROJECT))
for u in iter_units(PROJECT):
    print(f"{u.rel.ljust(w)}  {u.kind:<12} {(u.phase or '-'):<10} "
          f"{u.n_files:>7} files  {human(u.n_bytes)}")

print("\nre-planning to confirm idempotency...")
assert plan(PROJECT, CHECKOUT_PHASE) == [] or not APPLY, \
    "a second plan is non-empty after applying -- something is wrong"
print("a second run would move nothing." if APPLY else
      "(still a dry run -- set APPLY = True in Step 3 when ready.)")

## Done

Expected end state:

```
UNIT      KIND          PHASE       FILES  SIZE
data/raw  shared        -              20  67.0 MB
phase1_1  phase_folder  phase1_1      ...
phase1_2  phase_folder  phase1_2      ...
phase1_3  phase_folder  phase1_3      ...
```

Then drop `phase1_3_repo.zip` into `NeurIPS-CCAI-2026/phase1_3/` and run
`notebooks/phase1_3_cv.ipynb`. Its Step 2 searches Drive up to three levels
down, so it finds the cubes at the project root and the Phase 1.2 embeddings
inside `phase1_2/` without either being copied or modified.

If Step 4 reports a mismatch, stop. `before.json` holds the pre-move hashes and
every file is listed by path, so nothing is unrecoverable — but do not run
another move on top of it.